# Lab 6: Simple Tool‑Choosing Agent (Calculator vs RAG)
### *Build a tiny agent that decides when to call a tool — and test it like an engineer.*

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/labs/agents_lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Overview
In lecture, we defined an **agent** as a **control loop**:

> **Plan → Act (tool call) → Observe → Update → Stop**

In this lab, we’ll build a *minimal* agent that does **one loop step**:
1) decide whether to call a tool  
2) call it (Calculator or RAG)  
3) return an answer **plus a tool trace**

We’ll also practice a simple scientific workflow:
- pick a knob (**how you choose tools**)
- measure outcomes (**tool-choice accuracy** and a simple “quality” proxy)
- compare results to a hypothesis

> ⚠️ Scope: **No long-term memory yet.** No multi-day planning. Just tool selection.

---

## Learning goals
By the end of this lab, you can:
- Implement a **tool-selection policy** with simple Python logic.
- Call tools through **course-provided helper functions** (no API key in notebook).
- Log interactions as a **list of dictionaries**.
- Run a small **experiment** comparing two tool-selection policies.
- Build a small **Gradio demo** (mostly via helpers) to show your agent working.


In [1]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append('/content/main')

# Lab helpers live in course_utils.py
from course_utils import (
    lab6_setup,
    lab6_get_corpus,          # returns a small doc set for RAG
    lab6_build_retriever,     # builds a retriever given chunk_size, overlap, top_k
    lab6_rag_retrieve,        # retrieve passages for a query
    lab6_calculator,          # safe calculator tool
    lab6_generate_answer,     # (optional) LLM answer from question + context
    lab6_build_demo,          # Gradio demo builder
    lab6_default_eval_set     # small labeled questions for tool choice
)

lab6_setup()

print("✅ Environment ready!")


🔧 Setting up your environment...
  → Installing core packages...
installing mermaid-python
  → Installing additional packages: dspy
installing dspy
  → Setting random seed for reproducible results...
  → Checking API key...
🔑 Enter your OpenAI API key.
   (It will only be stored in this Colab runtime - it's safe!)
   Get your key from: https://platform.openai.com/api-keys
OpenAI API key: ··········
✅ API key set.
  → Adding course files to path...
✅ Setup complete!
✅ lab6_setup complete — ready.
✅ Environment ready!


## Pre-Lab Questions
Answer in **1–2 sentences each**. (Edit this cell and write below each question.)

1. When should an agent call a **calculator tool** instead of answering directly?
2. When should an agent use **RAG** (retrieval) instead of relying on the model’s memory?
3. What could go wrong if an agent loops without a stop condition?

**Your answers:**
1)  
2)  
3)


## Scientific Question & Hypothesis

### Scientific question (fill in the blanks)
**Question:**  
If we change **X =** the tool-selection policy (rules for choosing Calculator vs RAG vs None), what happens to  
**Y =** (1) tool-choice accuracy and (2) answer quality / groundedness?

### Your hypothesis (write your own!)
**My hypothesis:**  
I expect that a “smarter” policy that checks for **both** math patterns and doc-requests (e.g., “according to…”) will:
- increase tool-choice accuracy because _________  
- reduce mistakes like hallucinated arithmetic or unsupported doc claims because _________

Write your hypothesis here (2–4 sentences):

> **My hypothesis:**  
> …


---

## Scientific process plan (what we’ll do)
We’ll run a simple experiment:

1. **Question:** Does policy B beat policy A?
2. **Hypothesis:** You write it above.
3. **Experiment plan:**  
   - Keep the same evaluation questions.  
   - Run **policy A** and **policy B** on each question.  
   - Log tool choice + tool output + final answer.
4. **Measurement (metrics):**  
   - **Tool-choice accuracy** (how often did we pick the gold tool?)  
   - A simple “quality proxy”: did we return something reasonable? (We’ll use a lightweight rubric.)
5. **Conclusion shape:**  
   - “Policy B improved accuracy by ___ and reduced ___ failures” **OR**  
   - “No improvement; likely because ___ / dataset too small / heuristics too crude.”

Let’s build this step-by-step.


# Part 1 — Meet the tools (Calculator + RAG retrieval)
In this lab, **tools are just functions** you can call:

- `lab6_calculator(expression)` → returns `{"result": ...}` or `{"error": ...}`
- `lab6_rag_retrieve(query, retriever)` → returns `{"passages": [...]}`

We’ll start by trying them once so you know what their outputs look like.


In [2]:
# @title Try the calculator tool
lab6_calculator("17 * (2 + 3)")


{'result': 85}

In [3]:
# @title Build a retriever and try RAG retrieval
corpus = lab6_get_corpus()

# We can tune these knobs later (chunk_size, overlap, top_k)
chunk_size = 60
overlap = 15
top_k = 4

retriever = lab6_build_retriever(corpus=corpus, chunk_size=chunk_size, overlap=overlap, top_k=top_k)

out = lab6_rag_retrieve(query="According to the policy, can interns join on-call?", retriever=retriever)
out


{'passages': ['[policy_oncall::c0] the on call rotation is required for full time engineers and optional for interns interns may join on call only after completing onboarding and receiving manager approval interns should start with shadow shifts',
  '[policy_access::c0] interns are granted access to internal tools in the first week interns may not access customer production data elevated access requires manager approval',
  '[runbook_incidents::c0] responders should 1 acknowledge the page 2 assess severity 3 mitigate 4 communicate updates and 5 write a postmortem'],
 'scores': [0.6713681221008301, 0.4640411138534546, 0.19380579888820648]}

### Reflection
1–2 sentences:

- What does the RAG tool return?
- Why might those passages be helpful for answering “according to the policy…” questions?

Write here:


# Part 2 — Implement two tool-selection policies (with TODOs)

We’ll create two policies:

### Policy A (baseline)
A very simple rule:
- if the question contains digits and math operators → Calculator  
- else → None (answer directly)

### Policy B (improved)
A slightly smarter rule:
- if it looks like math → Calculator  
- else if it looks like “docs/policy/runbook” → RAG  
- else → None

Your job: implement the helper functions below.

> ✅ Keep it simple: students in this class are not expected to build a perfect classifier.  
> We’re practicing engineering iteration + measurement.


In [4]:
# @title ✅ TODO: Implement tool-selection helpers
import re

def looks_like_math(question: str) -> bool:
    """
    Return True if the question looks like it contains a math expression.

    Hints:
    - math questions often contain digits (0-9)
    - and math operators like + - * / % ( ) .

    TODO: implement a simple rule-based check.
    """
    # TODO: replace the next line with your code
    raise NotImplementedError("Implement looks_like_math(question)")

def looks_like_docs_question(question: str) -> bool:
    """
    Return True if the question sounds like it wants evidence from docs/policy/runbook.

    Hints:
    - Look for keywords like:
      'according to', 'policy', 'runbook', 'docs', 'documentation', 'source'

    TODO: implement a simple keyword check.
    """
    # TODO: replace the next line with your code
    raise NotImplementedError("Implement looks_like_docs_question(question)")

def choose_tool_policy_A(question: str) -> str:
    """
    Policy A:
      - math -> 'calculator'
      - else -> 'none'
    """
    # TODO: implement
    raise NotImplementedError("Implement choose_tool_policy_A(question)")

def choose_tool_policy_B(question: str) -> str:
    """
    Policy B:
      - math -> 'calculator'
      - docs -> 'rag'
      - else -> 'none'
    """
    # TODO: implement
    raise NotImplementedError("Implement choose_tool_policy_B(question)")


In [5]:
# @title (Optional) Reference solution for tool-choice helpers
# If you're stuck, run this cell to see one reasonable implementation.
# You can still improve it!

def looks_like_math(question: str) -> bool:
    has_digit = bool(re.search(r"\d", question))
    has_op = bool(re.search(r"[+\-*/%()]", question))
    return has_digit and has_op

def looks_like_docs_question(question: str) -> bool:
    q = question.lower()
    triggers = ["according to", "policy", "runbook", "docs", "documentation", "source", "what does it say"]
    return any(t in q for t in triggers)

def choose_tool_policy_A(question: str) -> str:
    return "calculator" if looks_like_math(question) else "none"

def choose_tool_policy_B(question: str) -> str:
    if looks_like_math(question):
        return "calculator"
    if looks_like_docs_question(question):
        return "rag"
    return "none"

print("Loaded reference solution ✅")


Loaded reference solution ✅


# Part 3 — Build a one-step agent + logging
Now we’ll build a *one-step agent*:

1) Choose tool (using a policy)
2) Call tool (if any)
3) Produce a final answer
4) Log what happened (for analysis)

We’ll keep “answer generation” simple and rely on course helpers:
- If tool is Calculator → we return the tool result in a sentence
- If tool is RAG → we retrieve passages and (optionally) ask the model to answer using the passages
- If no tool → we ask the model for a short answer

In real products, you’d always log:
- tool chosen
- tool inputs/outputs
- model prompt / response
- latency/cost

We’ll log a simplified version.


In [6]:
# @title ✅ TODO: Implement a simple agent_step() that logs tool choice and output
from typing import Callable, Dict, Any, List

def extract_math_expression(question: str) -> str:
    """Extract a rough math expression from the question (good enough for this lab)."""
    # keep digits, operators, parentheses, decimal points, and spaces
    expr = re.sub(r"[^0-9+\-*/().% ]", "", question)
    return expr.strip()

def agent_step(question: str,
               policy: Callable[[str], str],
               retriever,
               use_llm_for_final_answer: bool = True) -> Dict[str, Any]:
    """
    Run ONE agent step:
      - choose a tool using policy(question)
      - call the tool (if needed)
      - return a dict with tool, tool_output, answer, and trace

    TODO:
    - Implement the logic for the three cases: calculator, rag, none
    - Include a 'trace' list like ['TOOL=rag'] etc.
    - Always return a dict with keys:
        'question', 'tool', 'tool_output', 'answer', 'trace'
    """
    # TODO: choose tool
    raise NotImplementedError("Implement agent_step(...)")

# Helper: run agent over a list of questions and log results
def run_agent(questions: List[str], policy, retriever, use_llm_for_final_answer=True):
    logs = []
    for q in questions:
        out = agent_step(q, policy=policy, retriever=retriever, use_llm_for_final_answer=use_llm_for_final_answer)
        logs.append(out)
    return logs


### Tip: minimal “answer styles”
To keep your agent grounded and easy to debug:
- If you used **RAG**, consider answering with:
  - a short answer + **1 quote** from the top passage
- If you used **calculator**, include the expression you computed

That makes it much easier to spot mistakes.


# Part 4 — Experiment: Policy A vs Policy B

Now we’ll do the scientific loop.

We will:
1) Load a small evaluation set with gold tool labels
2) Run both policies on the same questions
3) Measure tool-choice accuracy
4) Inspect mistakes (qualitative debugging)

> You may find your policy gets 100% on this tiny dataset. That’s okay — your job is to **analyze** and propose how to make evaluation more realistic.


In [7]:
# @title Build a retriever for this lab (knobs you can change)
corpus = lab6_get_corpus()

chunk_size = 60
overlap = 15
top_k = 4

retriever = lab6_build_retriever(corpus=corpus, chunk_size=chunk_size, overlap=overlap, top_k=top_k)

eval_set = lab6_default_eval_set()
print("Loaded eval questions:", len(eval_set))
eval_set[:2]


Loaded eval questions: 8


[{'q': 'What is 17% of 84? Use arithmetic.', 'gold_tool': 'calculator'},
 {'q': 'Compute (12+5)*3.', 'gold_tool': 'calculator'}]

In [8]:
# @title ✅ TODO: Compute tool-choice accuracy for a set of logged interactions
import pandas as pd

def tool_choice_accuracy(logs, gold_labels):
    """
    Compute accuracy = (# correct tool choices) / (total)

    Inputs:
      logs: list of dicts from run_agent (each must include 'tool')
      gold_labels: list of gold tool strings (same length)

    TODO: implement and return a float between 0 and 1.
    """
    # TODO
    raise NotImplementedError("Implement tool_choice_accuracy(logs, gold_labels)")

def summarize_logs(logs, gold_labels):
    """Return a small DataFrame for easy inspection."""
    rows = []
    for out, gold in zip(logs, gold_labels):
        rows.append({
            "question": out.get("question"),
            "pred_tool": out.get("tool"),
            "gold_tool": gold,
            "correct": out.get("tool") == gold,
            "answer_preview": (out.get("answer","")[:120] + "…") if out.get("answer") else ""
        })
    return pd.DataFrame(rows)

# Build question list + gold labels
questions = [ex["q"] for ex in eval_set]
gold_tools = [ex["gold_tool"] for ex in eval_set]

print("Example question:", questions[0], "| gold:", gold_tools[0])


Example question: What is 17% of 84? Use arithmetic. | gold: calculator


In [9]:
# @title Run the experiment: Policy A vs Policy B
# NOTE: This cell assumes you implemented:
# - choose_tool_policy_A
# - choose_tool_policy_B
# - agent_step
# - tool_choice_accuracy

logs_A = run_agent(questions, policy=choose_tool_policy_A, retriever=retriever, use_llm_for_final_answer=True)
logs_B = run_agent(questions, policy=choose_tool_policy_B, retriever=retriever, use_llm_for_final_answer=True)

acc_A = tool_choice_accuracy(logs_A, gold_tools)
acc_B = tool_choice_accuracy(logs_B, gold_tools)

print("Policy A accuracy:", round(acc_A, 3))
print("Policy B accuracy:", round(acc_B, 3))

dfA = summarize_logs(logs_A, gold_tools)
dfB = summarize_logs(logs_B, gold_tools)

display(dfA)


NotImplementedError: Implement agent_step(...)

### Reflection (debugging like an engineer)
Pick **one mistake** and answer:

- Why do you think your policy chose the wrong tool?
- What small rule change might fix it?
- What new test question would you add to prevent regression?

Write here:


# Part 5 — A simple “quality proxy” metric (lightweight)
Tool-choice accuracy is important, but not the whole story.

For example:
- If you choose **calculator** correctly but extract the wrong expression → answer is wrong.
- If you choose **RAG** correctly but retrieved passages are irrelevant → answer might be unsupported.

We’ll add a tiny “quality proxy” metric:

- For calculator answers: did we produce a numeric result (not an error)?
- For RAG answers: did we retrieve at least 1 passage?
- For none: did we produce a non-empty answer?

This isn’t perfect, but it pushes us to think beyond one metric.


In [10]:
# @title ✅ TODO: Implement a simple quality_proxy_score
def quality_proxy_score(logs):
    """
    Return the fraction of examples that produced a "reasonable" output.

    Rules (simple):
      - tool == 'calculator': tool_output should have key 'result'
      - tool == 'rag': tool_output should have a non-empty 'passages' list
      - tool == 'none': answer should be a non-empty string

    TODO: implement and return a float between 0 and 1.
    """
    # TODO
    raise NotImplementedError("Implement quality_proxy_score(logs)")

# After you implement, try:
# print("Quality proxy A:", quality_proxy_score(logs_A))
# print("Quality proxy B:", quality_proxy_score(logs_B))


### Reflection
> Do Policy A and Policy B differ more on **accuracy** or on **quality proxy**?  
> What does that tell you about where to improve next?

Write here:


# Part 6 (Optional) — Gradio demo: Show your agent
In the lectures, we used Gradio to quickly demo a system.

Here, we’ll build a small app that:
- takes a question
- runs your agent step
- shows the chosen tool and tool trace
- shows the answer

✅ We’ll use `lab6_build_demo(...)` so you don’t have to write UI code from scratch.

**TODO:** pass in your policy function and your retriever.


In [11]:
# @title ✅ TODO: Launch the Gradio demo
# This assumes your choose_tool_policy_B and agent_step are implemented.

demo = lab6_build_demo(
    policy_fn=choose_tool_policy_B,
    agent_step_fn=agent_step,
    retriever=retriever
)

demo.launch(debug=False, share=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

---

## 🔹 Optional Extension – RAG as a Tool with DSPy ReAct

**Connection to Project:** Many projects combine RAG with agents using DSPy ReAct.

**Task:** Build a DSPy ReAct agent that uses RAG as a tool.



In [12]:
# @title Step 1: Setup DSPy

import dspy
dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"))

In [18]:
# @title Step 2: Define RAG as a Tool Function

def rag_tool(query: str) -> str:
    """Answer questions using RAG over your documents."""
    # Use the retriever from Part 1
    retrieved = lab6_rag_retrieve(query, retriever)
    passages = retrieved.get("passages", [])
    context = "\n\n".join(passages)

    # Optionally generate answer with LLM
    if context:
        answer = lab6_generate_answer(question=query, passages=context)
        return answer
    return "No relevant passages found."

In [19]:
# @title Step 3: Create ReAct Agent
# Create ReAct agent with RAG tool
# The agent will automatically decide when to use RAG
react_agent = dspy.ReAct(
    signature="question -> answer",
    tools=[rag_tool],
    max_iters=5
)

# Use it - the agent decides whether to use RAG or answer directly
result = react_agent(question="What does the policy say about interns and on-call?")
print(result.answer)

# Inspect what the agent did
dspy.inspect_history()  # See the reasoning and tool calls

The policy states that on-call rotation is required for full-time engineers and is optional for interns. Interns may join on-call duties after completing onboarding and obtaining manager approval, starting with shadow shifts.




[2026-01-10T19:04:54.474848]

System message:

Your input fields are:
1. `question` (str): 
2. `trajectory` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
What does the policy say about interns and on-call?

[[ ## trajectory ## ]]
[[ ## thought_0 ## ]]
I need to gather information regarding the policy related to interns and on-call d

In [21]:
# @title Step 4: Add Multiple Tools (Optional)

# You can combine RAG with other tools:
def calculator_tool(expression: str) -> str:
    """Perform calculations."""
    result = lab6_calculator(expression)
    return str(result.get("result", result.get("error", "Unknown error")))

# Agent with multiple tools
agent = dspy.ReAct(
    signature="question -> answer",
    tools=[rag_tool, calculator_tool],
    max_iters=5
)

# Test it
result = agent(question="What is 2+2?")  # Will use calculator
display(result)
result2 = agent(question="According to the policy, can interns join on-call?")  # Will use RAG
display(result2)



Prediction(
    trajectory={'thought_0': 'I need to perform a simple arithmetic calculation to find the answer to the question "What is 2+2?".', 'tool_name_0': 'calculator_tool', 'tool_args_0': {'expression': '2 + 2'}, 'observation_0': '4', 'thought_1': 'I have obtained the answer to the calculation, which is 4. Now I can finish the task since all information necessary for producing the answer is available.', 'tool_name_1': 'finish', 'tool_args_1': {}, 'observation_1': 'Completed.'},
    reasoning='The question asks for the result of a basic arithmetic operation, specifically the sum of 2 and 2. I used a calculator to compute this expression, which confirmed that the result is 4.',
    answer='4'
)

Prediction(
    trajectory={'thought_0': 'I need to check the policy documents to see if there are any specific guidelines regarding interns joining on-call duties. This might be detailed in the HR or internship policy documents, so I will perform a search for relevant information.', 'tool_name_0': 'rag_tool', 'tool_args_0': {'query': 'Can interns join on-call according to the policy?'}, 'observation_0': 'Interns may join on-call only after completing onboarding and receiving manager approval.', 'thought_1': 'The policy states that interns can join on-call after completing onboarding and receiving manager approval. I now have the necessary information to answer the question.', 'tool_name_1': 'finish', 'tool_args_1': {}, 'observation_1': 'Completed.'},
    reasoning='The policy specifies that interns are allowed to join on-call duties, but this is contingent upon them having completed their onboarding process and having received approval from their manager. This ensures that they are ad

**Why DSPy ReAct?**
- **Simple**: Just define functions as tools
- **Automatic**: Agent decides which tool to use
- **Inspectable**: Use `dspy.inspect()` to see reasoning
- **Consistent**: Uses the same DSPy patterns from Week 3

**Reference:** [DSPy ReAct Documentation](https://dspy.ai/api/modules/ReAct/)

**For your project:** This pattern will help you build agents that combine RAG with other tools!

---


---

## Results
Briefly summarize what you observed. Include:
- tool-choice accuracy for Policy A and Policy B
- quality proxy for Policy A and Policy B (after you implement it)
- 1–2 example questions where the policy surprised you

Write here:


## Conclusion
Tie back to your hypothesis:

- Was your hypothesis supported? Why or why not?
- How did changing your policy (X) affect tool-choice accuracy / quality (Y)?
- What would you try next to make your agent more reliable?

Write here:


## Post-Lab Reflection
Answer briefly (2–4 sentences each). (Edit this cell.)

1. Describe one question where your agent behaved in a surprising way.
2. What is one real-world situation where you would *trust* a tool-choosing agent more than a single model call?
3. What is one new question you have about agents after this lab?

Your answers:
1)  
2)  
3)


---

## 🧠 AI Usage Log

> Use this section to document any generative AI assistance (e.g., ChatGPT, Claude, Copilot) you used while completing this lab or assignment.  
> Be specific — transparency and reflection matter more than the amount of AI use.


| Tool Used | Purpose | Prompt / Context | Verification & Edits |
|------------|----------|------------------|----------------------|
| (e.g., ChatGPT (GPT-5)) | (e.g., debugging, code explanation, idea generation) | (e.g., "Why does my cosine similarity return NaN?") | (e.g., ran tests on sample input, compared with lecture code) |
| (Add rows as needed) | | | |

**Summary (2–3 sentences):**  
Briefly describe what you learned or how AI helped you think through the problem.  
Example: *AI helped me notice an off-by-one error in my indexing. I double-checked by printing intermediate results and confirmed the fix.*

---



In [ ]:
# @title ✅ Run checks for Lab I – L6
print("Running checks...")

# 1) looks_like_math should return booleans and detect simple math-ish strings
try:
    v = looks_like_math("Compute (12+5)*3")
    assert isinstance(v, bool)
    assert looks_like_math("2+2") is True
    assert looks_like_math("What is two plus two?") in [False, True]  # heuristic may vary
    print("✅ looks_like_math basic checks passed.")
except Exception as e:
    print("❌ looks_like_math check failed:", e)

# 2) looks_like_docs_question should detect 'according to' or 'policy'
try:
    v = looks_like_docs_question("According to the policy, can interns join on-call?")
    assert isinstance(v, bool)
    assert looks_like_docs_question("According to the policy...") is True
    print("✅ looks_like_docs_question basic checks passed.")
except Exception as e:
    print("❌ looks_like_docs_question check failed:", e)

# 3) choose_tool policies should return one of the allowed strings
try:
    for fn in [choose_tool_policy_A, choose_tool_policy_B]:
        t = fn("What is 2+2?")
        assert t in ["calculator", "rag", "none"]
    print("✅ choose_tool_policy_* basic checks passed.")
except Exception as e:
    print("❌ choose_tool_policy_* check failed:", e)

# 4) agent_step should return required keys (no network-heavy assertions)
try:
    corpus = lab6_get_corpus()
    retr = lab6_build_retriever(corpus=corpus, chunk_size=60, overlap=15, top_k=3)
    out = agent_step("What is 2+2?", policy=choose_tool_policy_B, retriever=retr, use_llm_for_final_answer=False)
    for k in ["question", "tool", "tool_output", "answer", "trace"]:
        assert k in out
    assert out["tool"] in ["calculator", "rag", "none"]
    assert isinstance(out["trace"], list)
    print("✅ agent_step output-format checks passed.")
except Exception as e:
    print("❌ agent_step check failed:", e)

# 5) tool_choice_accuracy should work on a tiny synthetic case
try:
    fake_logs = [{"tool": "calculator"}, {"tool": "none"}, {"tool": "rag"}]
    gold = ["calculator", "rag", "rag"]
    acc = tool_choice_accuracy(fake_logs, gold)
    assert abs(acc - (2/3)) < 1e-6
    print("✅ tool_choice_accuracy synthetic check passed.")
except Exception as e:
    print("❌ tool_choice_accuracy check failed:", e)

# 6) quality_proxy_score synthetic check
try:
    fake = [
        {"tool":"calculator", "tool_output":{"result":4}, "answer":"4"},
        {"tool":"calculator", "tool_output":{"error":"bad"}, "answer":"err"},
        {"tool":"rag", "tool_output":{"passages":["p1"]}, "answer":"..."},
        {"tool":"rag", "tool_output":{"passages":[]}, "answer":"..."},
        {"tool":"none", "tool_output":None, "answer":"hello"},
        {"tool":"none", "tool_output":None, "answer":""},
    ]
    q = quality_proxy_score(fake)
    assert abs(q - (3/6)) < 1e-6
    print("✅ quality_proxy_score synthetic check passed.")
except Exception as e:
    print("❌ quality_proxy_score check failed:", e)

print("Done.")


Running checks...
❌ looks_like_math check failed: name 'looks_like_math' is not defined
❌ looks_like_docs_question check failed: name 'looks_like_docs_question' is not defined
❌ choose_tool_policy_* check failed: name 'choose_tool_policy_A' is not defined
❌ agent_step check failed: name 'choose_tool_policy_B' is not defined
❌ tool_choice_accuracy check failed: Implement tool_choice_accuracy(logs, gold_labels)
❌ quality_proxy_score check failed: Implement quality_proxy_score(logs)
Done.
